In [1]:
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Assuming your notebook and CSV are in 'My Drive/Colab Notebooks' or a similar path
# You might need to adjust the path to your CSV file
folder_path = '/content/drive/MyDrive/Colab Notebooks/Mobapps-Task/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
apps=pd.read_csv(folder_path+'data/apps.csv')
apps.head()

,app_id,bundle_id,app_name,platform,status,category,subcategory_1,subcategory_2,subcategory_3,subcategory_4,size_mb,age_rating,avg_rating,total_ratings,advertiser_id,app_date_inserted
0,a_0000,com.maple.app0000,Bazaar_0000,ios,paused,Shopping,Coupons,Grocery,NaN,NaN,64.9,4+,3.22,491,adv_020,2025-04-06
1,a_0001,com.atlas.app0001,Kingdom_0001,ios,active,Games,Sports Games,Action,Arcade,NaN,83.7,12+,4.57,12157,adv_074,2025-12-23
2,a_0002,com.lumen.app0002.android,Money_0002,android,active,Finance,NaN,NaN,NaN,NaN,119.4,9+,4.30,5261,adv_027,2026-03-18
3,a_0003,com.orbit.app0003.android,Pro_0003,android,active,Utilities,Cleaner,VPN,NaN,NaN,94.6,4+,3.45,754,adv_001,2024-08-09
4,a_0004,com.indigo.app0004,Route_0004,ios,active,Travel,Public Transit,Hotels,Maps,NaN,107.5,17+,4.17,1266,adv_045,2026-03-19


In [4]:
# app features = {platform, status, category, subcategory_1, subcategory_2, subcategory_3, subcategory_4, size_mb, age_rating, avg_rating, total_ratings, app_date_inserted, how often the app gets clicked, how often the app gets installed, how often the app gets registered, how often the app gets leveled, how often the app gets opened, how often the app gets purchased, how often the app gets viewed,

#                 how long the app gets interacted on average, how long the app gets interacted on median,
#                 how long the app gets clicked on average, how long the app gets installed on average, how long the app gets leveled on average, how long the app gets registered on average, how long the app gets opened on average, how long the app gets purchased on average, how long the app gets viewed on average,
#                 how long the app gets clicked on median, how long the app gets installed on median, how long the app gets leveled on median, how long the app gets registered on median, how long the app gets opened on median, how long the app gets purchased on median, how long the app gets viewed on median,

#                 how often the app gets impressed,

#                 how often the app gets clicked,

#                 }

In [5]:
inters=pd.read_csv(folder_path+'data/interactions.csv')[['user_id','app_id','appsflyer_id','event_time','event_name','duration_seconds','media_source','attributed_touch_type']]
test=inters[inters['event_time']>='2026-04-23']
print(test.shape[0]/inters.shape[0])
inters = inters[inters['event_time']<'2026-04-23']
inters.head()

0.6877923741083481 0.3122076258916518


,user_id,app_id,appsflyer_id,event_time,event_name,duration_seconds,media_source,attributed_touch_type
0,u_003379,a_0302,1604819600815-6370015,2026-03-07T10:08:28.128658,af_click,22.0,snapchat_int,click
2,u_004219,a_0085,1536831578872-8560116,2026-04-02T09:52:40.084674,af_open,NaN,unityads_int,impression
3,u_003424,a_0318,1558962098531-4695611,2026-04-18T20:23:05.770141,af_click,17.0,googleadwords_int,click
4,u_000795,a_0063,1587127306210-3217181,2026-02-11T08:00:38.576117,af_open,182.0,ironsrc_int,impression
5,u_003711,a_0437,1570955095413-2580096,2026-04-04T03:15:59.765670,af_install,164.0,googleadwords_int,impression


In [6]:
users=pd.read_csv(folder_path+'data/users.csv')
users.head()

,user_id,customer_user_id,appsflyer_id,advertising_id,idfa,idfv,android_id,platform,device_model,os_version,country_code,language,carrier,signup_date,age_bucket
0,u_000000,u_000000,1666629883315-7423388,NaN,NaN,BDD640FB-0667-1AD1-1C80-317FA3B1799D,NaN,ios,"iPhone14,2",17.5,FR,fr,Vodafone,2025-03-19,25-34
1,u_000001,u_000001,1563273432170-5304572,NaN,23B8C1E9-3924-56DE-3EB1-3B9046685257,BD9C66B3-AD3C-2D6D-1A3D-1FA7BC8960A9,NaN,ios,"iPhone14,2",17.2,US,en,NaN,2025-07-31,25-34
2,u_000002,u_000002,1586569440870-3234489,NaN,NaN,NaN,8b9d2434e465e150,android,SM-G998B,15.0,IN,en,T-Mobile,2025-05-06,25-34
3,u_000003,u_000003,1577738800342-8204212,NaN,NaN,0822E8F3-6C03-1199-972A-846916419F82,NaN,ios,"iPad13,1",16.5,FR,fr,Deutsche Telekom,2025-10-20,25-34
4,u_000004,u_000004,1650937463655-5472471,3B8FAA18-37F8-A88B-17FC-695A07A0CA6E,NaN,NaN,9a1de644815ef6d1,android,Redmi Note 12,15.0,DE,de,Rogers,2025-12-21,45-54


In [31]:
event_type_mapping = {
    'af_click': 'click',
    'af_install': 'install',
    'af_complete_registration': 'register',
    'af_level_achieved': 'level',
    'af_open': 'open',
    'af_purchase': 'purchase',
    'af_view': 'view'
}
inters['interaction_type'] = inters['event_name'].map(event_type_mapping)
test['interaction_type'] = test['event_name'].map(event_type_mapping)


In [8]:
# Calculate how often each app gets each event type
app_event_counts = inters.groupby(['app_id', 'interaction_type']).size().unstack(fill_value=0)
app_event_counts.columns = [f'app_event_count_{col}' for col in app_event_counts.columns]

# Merge these new features into the 'apps' dataframe
apps = apps.merge(app_event_counts, on='app_id', how='left').fillna(0)

In [9]:
# Calculate overall average and median interaction duration per app
app_overall_duration_avg = inters.dropna(subset=['duration_seconds']).groupby('app_id')['duration_seconds'].mean().rename('overall_duration_avg')
app_overall_duration_median = inters.dropna(subset=['duration_seconds']).groupby('app_id')['duration_seconds'].median().rename('overall_duration_median')

# Calculate average duration per app for each interaction type
app_event_duration_avg = inters.dropna(subset=['duration_seconds']).groupby(['app_id', 'interaction_type'])['duration_seconds'].mean().unstack(fill_value=0)
app_event_duration_avg.columns = [f'duration_avg_{col}' for col in app_event_duration_avg.columns]

# Calculate median duration per app for each interaction type
app_event_duration_median = inters.dropna(subset=['duration_seconds']).groupby(['app_id', 'interaction_type'])['duration_seconds'].median().unstack(fill_value=0)
app_event_duration_median.columns = [f'duration_median_{col}' for col in app_event_duration_median.columns]

# Merge all new duration features into the apps dataframe
apps = apps.merge(app_overall_duration_avg, on='app_id', how='left')
apps = apps.merge(app_overall_duration_median, on='app_id', how='left')
apps = apps.merge(app_event_duration_avg, on='app_id', how='left')
apps = apps.merge(app_event_duration_median, on='app_id', how='left')


In [10]:
# Filter interactions that are impressions
impressions_df = inters[inters['attributed_touch_type'] == 'impression'].copy()

# how often the app gets impressed
app_impression_count = impressions_df.groupby('app_id').size().rename('impression_count')

# Merge all new impression features into the 'apps' dataframe
apps = apps.merge(app_impression_count, on='app_id', how='left').fillna(0)

In [11]:
# 1. How often the app gets clicked (total clicks)
app_total_clicks = inters[inters['interaction_type'] == 'click'].groupby('app_id').size().rename('total_clicks')

# 2. How often the app gets installed (total installs)
app_total_installs = inters[inters['interaction_type'] == 'install'].groupby('app_id').size().rename('total_installs')

# 3. How often the app gets purchased (total purchases)
app_total_purchases = inters[inters['interaction_type'] == 'purchase'].groupby('app_id').size().rename('total_purchases')

# Merge all new features into the 'apps' dataframe
apps = apps.merge(app_total_clicks, on='app_id', how='left').fillna(0)
apps = apps.merge(app_total_installs, on='app_id', how='left').fillna(0)
apps = apps.merge(app_total_purchases, on='app_id', how='left').fillna(0)

In [12]:
apps['status'].value_counts(normalize=True)

,proportion
status,
active,0.758
blocked,0.132
paused,0.110


In [13]:
# open, 0.1
# view, 0.15
# click, 0.2
# install, 0.2
# register 0.3
# level achieve, 0.4
# purchase, 0.6


In [14]:
# inters features = {media_source, attributed_touch_type, interaction level (view/click/install/open/register/level/purchase)}

In [15]:
inters.groupby(['interaction_type','event_name']).size()

,,0
interaction_type,event_name,
click,af_click,73780
install,af_install,16902
level,af_level_achieved,6648
open,af_open,67358
purchase,af_purchase,16789
register,af_complete_registration,13528
view,af_view,142276


In [16]:
# # users features = {platform, iphone/android, phone_company, country_code, language, signup_date,age_bucket, how often client clicks, how often client installs, how often client registers, how often client levels, how often client opens, how often client purchases, how often client views,

#                   how long the user interacts on average, how long the user interacts on median, how long the user interacts by click on average, how long the user interacts by install on average, how long the user interacts by registration on average, how long the user interacts by leveling on average, how long the user interacts by opening on average, how long the user interacts by purchasing on average, how long the user interacts by viewing on average,
#                   how long the user interacts by click on median, how long the user interacts by install on median, how long the user interacts by registration on median, how long the user interacts by leveling on median, how long the user interacts by opening on median, how long the user interacts by purchasing on median, how long the user interacts by viewing on median,

#                   how often client had impression,

#                   how often client had clicked,

#                   }

In [17]:
# Calculate overall average and median interaction duration per user
user_overall_duration_avg = inters.dropna(subset=['duration_seconds']).groupby('user_id')['duration_seconds'].mean().rename('user_overall_duration_avg')
user_overall_duration_median = inters.dropna(subset=['duration_seconds']).groupby('user_id')['duration_seconds'].median().rename('user_overall_duration_median')

# Merge all new duration features into the 'users' dataframe
users = users.merge(user_overall_duration_avg, on='user_id', how='left').fillna(0)
users = users.merge(user_overall_duration_median, on='user_id', how='left').fillna(0)

In [18]:
# Filter interactions that are impressions
user_impressions_df = inters[inters['attributed_touch_type'] == 'impression'].copy()

# How often client had impression
user_impression_count = user_impressions_df.groupby('user_id').size().rename('user_impression_count')

# Merge all new impression features into the 'users' dataframe
users = users.merge(user_impression_count, on='user_id', how='left').fillna(0)


In [19]:
# 1. How often client had clicked (total clicks)
user_total_clicks = inters[inters['interaction_type'] == 'click'].groupby('user_id').size().rename('user_total_clicks')

# 2. How often the user installs (total installs)
user_total_installs = inters[inters['interaction_type'] == 'install'].groupby('user_id').size().rename('user_total_installs')

# 3. How often the user purchases (total purchases)
user_total_purchases = inters[inters['interaction_type'] == 'purchase'].groupby('user_id').size().rename('user_total_purchases')


# Merge all new features into the 'users' dataframe
users = users.merge(user_total_clicks, on='user_id', how='left').fillna(0)
users = users.merge(user_total_installs, on='user_id', how='left').fillna(0)
users = users.merge(user_total_purchases, on='user_id', how='left').fillna(0)


In [20]:
# Calculate how often each client performs various actions
user_event_counts = inters.groupby(['user_id', 'interaction_type']).size().unstack(fill_value=0)
user_event_counts.columns = [f'user_event_count_{col}' for col in user_event_counts.columns]

# Merge these new features into the 'users' dataframe
users = users.merge(user_event_counts, on='user_id', how='left').fillna(0)

In [21]:
users['device_model'].unique()

array(['iPhone14,2', 'SM-G998B', 'iPad13,1', 'Redmi Note 12',
       'iPhone15,3', 'Pixel 8', 'Mi 13', 'iPhone15,2', 'Pixel 7',
       'iPhone16,2', 'Pixel 8 Pro', 'OnePlus 11', 'iPhone16,1',
       'iPhone14,5', 'SM-S918B', 'SM-S908U'], dtype=object)

In [22]:
apps.drop(columns=['bundle_id','app_name'],inplace=True)
apps.head()

,app_id,platform,status,category,subcategory_1,subcategory_2,subcategory_3,subcategory_4,size_mb,age_rating,...,duration_median_install,duration_median_level,duration_median_open,duration_median_purchase,duration_median_register,duration_median_view,impression_count,total_clicks,total_installs,total_purchases
0,a_0000,ios,paused,Shopping,Coupons,Grocery,0,0,64.9,4+,...,89.0,133.0,134.0,90.0,101.0,9.0,455.0,355.0,78.0,79.0
1,a_0001,ios,active,Games,Sports Games,Action,Arcade,0,83.7,12+,...,69.0,250.0,105.0,165.0,188.0,11.0,196.0,148.0,36.0,33.0
2,a_0002,android,active,Finance,0,0,0,0,119.4,9+,...,79.5,199.0,148.0,153.0,98.0,11.0,190.0,136.0,27.0,36.0
3,a_0003,android,active,Utilities,Cleaner,VPN,0,0,94.6,4+,...,61.5,156.0,124.5,122.5,235.0,9.0,192.0,166.0,22.0,26.0
4,a_0004,ios,active,Travel,Public Transit,Hotels,Maps,0,107.5,17+,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
users.drop(columns=['customer_user_id','appsflyer_id','advertising_id','idfa','idfv','android_id'],inplace=True)
users.head()

,user_id,platform,device_model,os_version,country_code,language,carrier,signup_date,age_bucket,user_overall_duration_avg,...,user_total_clicks,user_total_installs,user_total_purchases,user_event_count_click,user_event_count_install,user_event_count_level,user_event_count_open,user_event_count_purchase,user_event_count_register,user_event_count_view
0,u_000000,ios,"iPhone14,2",17.5,FR,fr,Vodafone,2025-03-19,25-34,29.000000,...,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
1,u_000001,ios,"iPhone14,2",17.2,US,en,0,2025-07-31,25-34,54.571429,...,4.0,1.0,0.0,4.0,1.0,0.0,2.0,0.0,0.0,7.0
2,u_000002,android,SM-G998B,15.0,IN,en,T-Mobile,2025-05-06,25-34,16.000000,...,2.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,9.0
3,u_000003,ios,"iPad13,1",16.5,FR,fr,Deutsche Telekom,2025-10-20,25-34,64.132353,...,32.0,8.0,8.0,32.0,8.0,0.0,20.0,8.0,4.0,66.0
4,u_000004,android,Redmi Note 12,15.0,DE,de,Rogers,2025-12-21,45-54,84.566667,...,7.0,3.0,1.0,7.0,3.0,2.0,4.0,1.0,1.0,13.0


In [24]:
inters['event_time'] = pd.to_datetime(inters['event_time'], errors='coerce')
test['event_time'] = pd.to_datetime(test['event_time'], errors='coerce')
inters['event_time'].describe()

,event_time
count,337280
mean,2026-04-03 07:48:09.973175296
min,2026-01-30 00:00:27.116482
25%,2026-03-26 06:45:23.797953024
50%,2026-04-09 03:50:41.009378560
75%,2026-04-17 06:32:03.508368384
max,2026-04-22 23:59:55.031896


In [25]:
users['signup_date'] = pd.to_datetime(users['signup_date'], errors='coerce')
users['days_since_signup_at_event'] = (pd.to_datetime('2026-04-30') - users['signup_date']).dt.days
apps['app_date_inserted'] = pd.to_datetime(apps['app_date_inserted'])
apps['app_date_inserted'] = (pd.to_datetime('2026-04-30') - apps['app_date_inserted']).dt.days
users.drop(columns=['signup_date'],inplace=True)

In [26]:
# Convert categorical columns in 'apps' to 'category' dtype
apps_categorical_cols = ['platform', 'status', 'category', 'subcategory_1', 'subcategory_2', 'subcategory_3', 'subcategory_4', 'age_rating', 'advertiser_id']
for col in apps_categorical_cols:
    if col in apps.columns:
        apps[col] = apps[col].astype('category')

# Convert categorical columns in 'users' to 'category' dtype
users_categorical_cols = ['platform', 'device_model', 'os_version', 'country_code', 'language', 'carrier', 'age_bucket']
for col in users_categorical_cols:
    if col in users.columns:
        users[col] = users[col].astype('category')

print("Categorical columns in 'apps' after conversion:")
print(apps[apps_categorical_cols].dtypes)
print("\nCategorical columns in 'users' after conversion:")
print(users[users_categorical_cols].dtypes)

Categorical columns in 'apps' after conversion:
platform         category
status           category
category         category
subcategory_1    category
subcategory_2    category
subcategory_3    category
subcategory_4    category
age_rating       category
advertiser_id    category
dtype: object

Categorical columns in 'users' after conversion:
platform        category
device_model    category
os_version      category
country_code    category
language        category
carrier         category
age_bucket      category
dtype: object


In [28]:
train=inters

In [40]:
# Group by user_id and app_id, and count occurrences of each interaction_type
interaction_counts = train.groupby(['user_id', 'app_id', 'interaction_type']).size().unstack(fill_value=0).reset_index()

# Rename columns to match the requested formula variables
# Ensure all required columns exist, fill with 0 if not present in the interactions
required_cols = ['open', 'view', 'click', 'install', 'level', 'purchase']
for col in required_cols:
    if col not in interaction_counts.columns:
        interaction_counts[col] = 0

# Calculate the 'interaction_level' using the specified formula
# Note: 'levelachieve' in the prompt corresponds to 'level' in our interaction_type mapping
interaction_counts['interaction_level'] = (
    interaction_counts['open'] * 0.1 +
    interaction_counts['view'] * 0.15 +
    interaction_counts['click'] * 0.2 +
    interaction_counts['install'] * 0.2 +
    interaction_counts['register'] * 0.3 +
    interaction_counts['level'] * 0.4 + # 'n_levelachieve' from prompt is 'level' here
    interaction_counts['purchase'] * 0.6
)

# Create the new dataframe with only the requested columns
train_2 = interaction_counts[['user_id', 'app_id', 'interaction_level']]

train_2

interaction_type,user_id,app_id,interaction_level
0,u_000000,a_0241,0.20
1,u_000000,a_0278,0.20
2,u_000000,a_0379,0.15
3,u_000001,a_0087,0.30
4,u_000001,a_0160,0.15
...,...,...,...
235103,u_999994,a_0129,0.10
235104,u_999995,a_0321,0.15
235105,u_999996,a_0158,0.10
235106,u_999998,a_0079,0.40


In [41]:
# Group by user_id and app_id, and count occurrences of each interaction_type
interaction_counts = test.groupby(['user_id', 'app_id', 'interaction_type']).size().unstack(fill_value=0).reset_index()

# Rename columns to match the requested formula variables
# Ensure all required columns exist, fill with 0 if not present in the interactions
required_cols = ['open', 'view', 'click', 'install', 'level', 'purchase']
for col in required_cols:
    if col not in interaction_counts.columns:
        interaction_counts[col] = 0

# Calculate the 'interaction_level' using the specified formula
# Note: 'levelachieve' in the prompt corresponds to 'level' in our interaction_type mapping
interaction_counts['interaction_level'] = (
    interaction_counts['open'] * 0.1 +
    interaction_counts['view'] * 0.15 +
    interaction_counts['click'] * 0.2 +
    interaction_counts['install'] * 0.2 +
    interaction_counts['register'] * 0.3 +
    interaction_counts['level'] * 0.4 + # 'n_levelachieve' from prompt is 'level' here
    interaction_counts['purchase'] * 0.6
)

# Create the new dataframe with only the requested columns
test_2 = interaction_counts[['user_id', 'app_id', 'interaction_level']]

test_2

interaction_type,user_id,app_id,interaction_level
0,u_000000,a_0061,0.20
1,u_000000,a_0069,0.20
2,u_000000,a_0092,0.60
3,u_000001,a_0168,0.15
4,u_000001,a_0186,0.20
...,...,...,...
123534,u_999975,a_0172,0.60
123535,u_999976,a_0244,0.10
123536,u_999985,a_0065,0.10
123537,u_999989,a_0440,0.15


In [42]:
train_2=train_2.merge(users,how='left',on='user_id')
train_2=train_2.merge(apps,how='left',on='app_id')
test_2=test_2.merge(users,how='left',on='user_id')
test_2=test_2.merge(apps,how='left',on='app_id')
train_2

,user_id,app_id,interaction_level,platform_x,device_model,os_version,country_code,language,carrier,age_bucket,...,duration_median_install,duration_median_level,duration_median_open,duration_median_purchase,duration_median_register,duration_median_view,impression_count,total_clicks,total_installs,total_purchases
0,u_000000,a_0241,0.20,ios,"iPhone14,2",17.5,FR,fr,Vodafone,25-34,...,71.0,115.0,124.0,155.0,108.0,10.0,824.0,594.0,134.0,142.0
1,u_000000,a_0278,0.20,ios,"iPhone14,2",17.5,FR,fr,Vodafone,25-34,...,99.5,173.5,118.0,118.0,121.0,11.0,352.0,267.0,46.0,50.0
2,u_000000,a_0379,0.15,ios,"iPhone14,2",17.5,FR,fr,Vodafone,25-34,...,109.0,144.0,117.0,100.0,140.0,9.0,130.0,84.0,20.0,21.0
3,u_000001,a_0087,0.30,ios,"iPhone14,2",17.2,US,en,0,25-34,...,57.0,129.0,128.0,82.5,113.5,11.0,183.0,142.0,39.0,28.0
4,u_000001,a_0160,0.15,ios,"iPhone14,2",17.2,US,en,0,25-34,...,76.0,152.0,109.0,87.0,114.5,12.0,193.0,153.0,34.0,35.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235103,u_999994,a_0129,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,107.5,126.0,97.0,103.5,247.0,11.5,91.0,65.0,11.0,12.0
235104,u_999995,a_0321,0.15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,58.0,157.0,134.0,121.5,157.0,11.0,345.0,244.0,42.0,61.0
235105,u_999996,a_0158,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,53.0,70.0,127.0,131.0,147.0,7.0,74.0,58.0,5.0,11.0
235106,u_999998,a_0079,0.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,102.5,125.0,122.5,146.0,107.0,11.0,279.0,202.0,43.0,54.0


In [43]:
train_2 = train_2.sort_values('user_id').reset_index(drop=True)
test_2 = test_2.sort_values('user_id').reset_index(drop=True)

In [44]:
group_sizes = (
    train_2.groupby('user_id', sort=False)
    .size()
    .tolist()
)

In [45]:
def to_relevance(x):
    x = round(x, 2)

    if x == 0:
        return 0
    elif x <= 0.10:
        return 1
    elif x <= 0.15:
        return 2
    elif x <= 0.20:
        return 3
    elif x <= 0.30:
        return 4
    elif x <= 0.50:
        return 5
    elif x <= 0.80:
        return 6
    else:
        return 7

train_2['interaction_rank'] = train_2['interaction_level'].apply(to_relevance)
test_2['interaction_rank'] = test_2['interaction_level'].apply(to_relevance)
train_2['interaction_rank'].value_counts(normalize=True).sort_index()

,proportion
interaction_rank,
1,0.154333
2,0.325387
3,0.213723
4,0.096564
5,0.096568
6,0.082528
7,0.030896


In [46]:
predictors = [x for x in train_2.columns.to_list() if x not in (['user_id','app_id','event_time','interaction_level','interaction_rank'])]
# predictors = [
#     # user profile
#     'platform_x',
#     'country_code',
#     'language',
#     'age_bucket',

#     # general user activity
#     'user_total_clicks',
#     'user_total_installs',
#     'user_total_purchases',
#     'user_overall_duration_avg',

#     # app information
#     'status',
#     'category',
#     'subcategory_1',
#     'size_mb',
#     'age_rating',
#     'avg_rating',
#     'total_ratings',

#     # app popularity / engagement
#     'app_event_count_click',
#     'app_event_count_install',
#     'app_event_count_open',
#     'app_event_count_purchase',
# ]
target = 'interaction_rank'

In [47]:
cat_cols = [col for col in predictors if (train_2[col].dtype == 'object' or
                               train_2[col].dtype == 'category') and col not in ['user_id','app_id','interaction_rank','interaction_level']]
print("Categorical features in train_2:", cat_cols)

Categorical features in train_2: ['platform_x', 'device_model', 'os_version', 'country_code', 'language', 'carrier', 'age_bucket', 'platform_y', 'status', 'category', 'subcategory_1', 'subcategory_2', 'subcategory_3', 'subcategory_4', 'age_rating', 'advertiser_id']


In [48]:
from lightgbm import LGBMRanker

X_train = train_2[predictors]
y_train = train_2[target]

ranker = LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    label_gain=[0, 1, 2, 3, 4,5,6,7],
)

ranker.fit(
    X_train,
    y_train,
    group=group_sizes,
    categorical_feature=cat_cols
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.059072 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7682
[LightGBM] [Info] Number of data points in the train set: 235108, number of used features: 61


LGBMRanker(label_gain=[0, 1, 2, 3, 4, 5, 6, 7], learning_rate=0.05,
           metric='ndcg', n_estimators=300, objective='lambdarank',
           random_state=42)

In [49]:
all_users = pd.DataFrame({'user_id': users['user_id'].unique()})
all_apps = pd.DataFrame({'app_id': apps['app_id'].unique()})

test_3 = pd.merge(all_users.assign(key=1), all_apps.assign(key=1), on='key').drop('key', axis=1)

print("DataFrame with all possible user-app pairs:")
test_3

DataFrame with all possible user-app pairs:


,user_id,app_id
0,u_000000,a_0000
1,u_000000,a_0001
2,u_000000,a_0002
3,u_000000,a_0003
4,u_000000,a_0004
...,...,...
4999995,u_009999,a_0495
4999996,u_009999,a_0496
4999997,u_009999,a_0497
4999998,u_009999,a_0498


In [50]:
test_3=test_3.merge(users,how='left',on='user_id')
test_3=test_3.merge(apps,how='left',on='app_id')
test_3 = test_3.merge(test_2[['user_id','app_id','interaction_level','interaction_rank']],how='left',on=['user_id','app_id'])
test_3

,user_id,app_id,platform_x,device_model,os_version,country_code,language,carrier,age_bucket,user_overall_duration_avg,...,duration_median_open,duration_median_purchase,duration_median_register,duration_median_view,impression_count,total_clicks,total_installs,total_purchases,interaction_level,interaction_rank
0,u_000000,a_0000,ios,"iPhone14,2",17.5,FR,fr,Vodafone,25-34,29.000000,...,134.0,90.0,101.0,9.0,455.0,355.0,78.0,79.0,NaN,NaN
1,u_000000,a_0001,ios,"iPhone14,2",17.5,FR,fr,Vodafone,25-34,29.000000,...,105.0,165.0,188.0,11.0,196.0,148.0,36.0,33.0,NaN,NaN
2,u_000000,a_0002,ios,"iPhone14,2",17.5,FR,fr,Vodafone,25-34,29.000000,...,148.0,153.0,98.0,11.0,190.0,136.0,27.0,36.0,NaN,NaN
3,u_000000,a_0003,ios,"iPhone14,2",17.5,FR,fr,Vodafone,25-34,29.000000,...,124.5,122.5,235.0,9.0,192.0,166.0,22.0,26.0,NaN,NaN
4,u_000000,a_0004,ios,"iPhone14,2",17.5,FR,fr,Vodafone,25-34,29.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4999995,u_009999,a_0495,ios,"iPhone15,3",17.4,DE,de,Vodafone,25-34,48.809524,...,117.5,147.5,101.0,10.0,446.0,299.0,78.0,74.0,NaN,NaN
4999996,u_009999,a_0496,ios,"iPhone15,3",17.4,DE,de,Vodafone,25-34,48.809524,...,112.0,124.5,122.0,8.0,65.0,49.0,12.0,14.0,NaN,NaN
4999997,u_009999,a_0497,ios,"iPhone15,3",17.4,DE,de,Vodafone,25-34,48.809524,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
4999998,u_009999,a_0498,ios,"iPhone15,3",17.4,DE,de,Vodafone,25-34,48.809524,...,115.5,103.0,104.0,10.0,749.0,574.0,112.0,124.0,NaN,NaN


In [51]:
test_3['score'] = ranker.predict(
    test_3[predictors]
)

In [52]:
test_3[test_3['interaction_rank'].isnull()].shape

(4876602, 66)

In [53]:
# Group by user_id and get top 10 app_ids based on interaction_rank (real)
top10_real_apps = (
    test_3
    .dropna(subset=['interaction_rank'])
    .sort_values(
        ['user_id', 'interaction_rank'],
        ascending=[True, False]
    )
    .groupby('user_id')
    .head(10)
    .groupby('user_id')['app_id']
    .apply(list)
    .reset_index(name='top10_real')
)

# Group by user_id and get top 10 app_ids based on score (predicted)
top10_predicted_apps = test_3.groupby('user_id').apply(lambda x: x.sort_values(by='score', ascending=False).head(10)['app_id'].tolist()).reset_index(name='top10_predicted')

# Merge the two results into a single DataFrame
test_3_scored = pd.merge(top10_real_apps, top10_predicted_apps, on='user_id', how='outer')
test_3_scored['top10_real'] = test_3_scored['top10_real'].apply(
    lambda x: x if isinstance(x, list) else []
)

test_3_scored['top10_real_count'] = test_3_scored['top10_real'].apply(len)
test_3_scored['top10_real_count'].describe()

/tmp/ipykernel_11122/247554115.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top10_predicted_apps = test_3.groupby('user_id').apply(lambda x: x.sort_values(by='score', ascending=False).head(10)['app_id'].tolist()).reset_index(name='top10_predicted')


,top10_real_count
count,10000.000000
mean,6.299600
std,3.657674
min,0.000000
25%,3.000000
50%,7.000000
75%,10.000000
max,10.000000


In [54]:
test_3_scored

,user_id,top10_real,top10_predicted,top10_real_count
0,u_000000,"[a_0092, a_0061, a_0069]","[a_0127, a_0225, a_0126, a_0337, a_0211, a_039...",3
1,u_000001,"[a_0186, a_0316, a_0387, a_0168, a_0278, a_0484]","[a_0493, a_0241, a_0215, a_0245, a_0065, a_032...",6
2,u_000002,"[a_0180, a_0184, a_0326, a_0422, a_0172, a_020...","[a_0065, a_0327, a_0241, a_0000, a_0039, a_003...",8
3,u_000003,"[a_0271, a_0106, a_0241, a_0119, a_0191, a_027...","[a_0241, a_0327, a_0273, a_0245, a_0065, a_037...",10
4,u_000004,"[a_0039, a_0127, a_0037, a_0055, a_0111, a_004...","[a_0275, a_0273, a_0437, a_0037, a_0241, a_004...",10
...,...,...,...,...
9995,u_009995,[a_0216],"[a_0027, a_0365, a_0478, a_0000, a_0166, a_010...",1
9996,u_009996,"[a_0229, a_0495, a_0485, a_0070, a_0267, a_0494]","[a_0109, a_0269, a_0268, a_0037, a_0148, a_041...",6
9997,u_009997,"[a_0208, a_0209, a_0388, a_0168, a_0360, a_036...","[a_0327, a_0031, a_0273, a_0123, a_0410, a_000...",9
9998,u_009998,"[a_0178, a_0164, a_0369, a_0437, a_0045, a_005...","[a_0399, a_0257, a_0054, a_0245, a_0215, a_004...",10


In [55]:
import math

def recall_at_k(actual, predicted, k=10):
    actual = set(actual)

    if len(actual) == 0:
        return None

    predicted = predicted[:k]

    hits = sum(app in actual for app in predicted)

    return hits / len(actual)


def ndcg_at_k(actual, predicted, k=10):
    actual = set(actual)

    if len(actual) == 0:
        return None

    predicted = predicted[:k]

    # DCG
    dcg = 0

    for i, app in enumerate(predicted):
        if app in actual:
            dcg += 1 / math.log2(i + 2)

    # Ideal DCG
    ideal_hits = min(len(actual), k)

    idcg = sum(
        1 / math.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg

In [56]:
test_3_scored['recall@10'] = test_3_scored.apply(
    lambda row: recall_at_k(
        row['top10_real'],
        row['top10_predicted'],
        10
    ),
    axis=1
)

test_3_scored['ndcg@10'] = test_3_scored.apply(
    lambda row: ndcg_at_k(
        row['top10_real'],
        row['top10_predicted'],
        10
    ),
    axis=1
)
test_3_scored['hits@10'] = test_3_scored.apply(
    lambda row: len(
        set(row['top10_real']) &
        set(row['top10_predicted'][:10])
    ),
    axis=1
)

In [57]:
valid = test_3_scored[
    test_3_scored['top10_real_count'] > 0
]

print("Users evaluated:", len(valid))
print("Recall@10:", valid['recall@10'].mean())
print("NDCG@10:", valid['ndcg@10'].mean())

Users evaluated: 9422
Recall@10: 0.06479044317082949
NDCG@10: 0.06330865364475655


In [58]:
print(valid[['recall@10', 'ndcg@10', 'hits@10']].describe())

print(
    "Perfect Recall:",
    (valid['recall@10'] == 1).mean()
)

print(
    "Zero Recall:",
    (valid['recall@10'] == 0).mean()
)

         recall@10      ndcg@10      hits@10
count  9422.000000  9422.000000  9422.000000
mean      0.064790     0.063309     0.509658
std       0.109039     0.102816     0.757556
min       0.000000     0.000000     0.000000
25%       0.000000     0.000000     0.000000
50%       0.000000     0.000000     0.000000
75%       0.100000     0.098039     1.000000
max       1.000000     1.000000     5.000000
Perfect Recall: 0.0029717682020802376
Zero Recall: 0.6254510719592443


In [59]:
popular_top10 = (
    train['app_id']
    .value_counts()
    .head(10)
    .index
    .tolist()
)

popular_top10

['a_0273',
 'a_0327',
 'a_0065',
 'a_0377',
 'a_0241',
 'a_0245',
 'a_0498',
 'a_0477',
 'a_0323',
 'a_0275']

In [60]:


test_3_scored['top10_popularity'] = test_3_scored.apply(
    lambda _: popular_top10.copy(),
    axis=1
)
test_3_scored['pop_recall@10'] = test_3_scored.apply(
    lambda row: recall_at_k(
        row['top10_real'],
        row['top10_popularity'],
        k=10
    ),
    axis=1
)

test_3_scored['pop_ndcg@10'] = test_3_scored.apply(
    lambda row: ndcg_at_k(
        row['top10_real'],
        row['top10_popularity'],
        k=10
    ),
    axis=1
)

valid = test_3_scored[
    test_3_scored['top10_real_count'] > 0
]

comparison = pd.DataFrame({
    'model': ['LambdaMART', 'Popularity'],
    'Recall@10': [
        valid['recall@10'].mean(),
        valid['pop_recall@10'].mean()
    ],
    'NDCG@10': [
        valid['ndcg@10'].mean(),
        valid['pop_ndcg@10'].mean()
    ]
})

comparison

,model,Recall@10,NDCG@10
0,LambdaMART,0.064790,0.063309
1,Popularity,0.081566,0.071172


In [61]:
# ============================================================
# OFFICIAL-STYLE EVALUATION
# Recall@10 + NDCG@10
# + Popularity baseline
# ============================================================

import math
import pandas as pd


# ------------------------------------------------------------
# 1. DATAFRAMES
# ------------------------------------------------------------

train_interactions = train
test_interactions = test
predictions_df = test_3_scored


# ------------------------------------------------------------
# 2. ACTUAL FUTURE APPS
#
# Important:
# We do NOT take the 10 strongest apps.
# Every unique app appearing in the test period is relevant.
# ------------------------------------------------------------

actual_apps = (
    test_interactions
    .groupby('user_id')['app_id']
    .apply(lambda x: list(set(x)))
    .reset_index(name='actual_apps')
)


# ------------------------------------------------------------
# 3. MODEL PREDICTIONS
#
# Keep only user_id + ranked Top-10 predictions.
# ------------------------------------------------------------

model_predictions = predictions_df[
    ['user_id', 'top10_predicted']
].copy()


# Remove duplicate recommendations while preserving order
def unique_preserve_order(x):
    if not isinstance(x, list):
        return []

    return list(dict.fromkeys(x))


model_predictions['top10_predicted'] = (
    model_predictions['top10_predicted']
    .apply(unique_preserve_order)
)


# ------------------------------------------------------------
# 4. CREATE EVALUATION TABLE
#
# Start from actual test users.
# If model has no prediction for a user -> [] -> score 0.
# ------------------------------------------------------------

evaluation_df = actual_apps.merge(
    model_predictions,
    on='user_id',
    how='left'
)

evaluation_df['top10_predicted'] = (
    evaluation_df['top10_predicted']
    .apply(lambda x: x if isinstance(x, list) else [])
)


# ------------------------------------------------------------
# 5. METRIC FUNCTIONS
# Same logic as provided evaluate.py
# ------------------------------------------------------------

def recall_at_k(predicted, actual, k=10):

    actual = set(actual)

    if not actual:
        return 0.0

    top_k = predicted[:k]

    hits = sum(
        1 for item in top_k
        if item in actual
    )

    return hits / len(actual)


def ndcg_at_k(predicted, actual, k=10):

    actual = set(actual)

    if not actual:
        return 0.0

    # DCG
    dcg = 0.0

    for i, item in enumerate(predicted[:k]):

        if item in actual:
            dcg += 1.0 / math.log2(i + 2)


    # Ideal DCG
    ideal_hits = min(len(actual), k)

    idcg = sum(
        1.0 / math.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg if idcg > 0 else 0.0


# ------------------------------------------------------------
# 6. EVALUATE LAMBDAMART
# ------------------------------------------------------------

evaluation_df['recall@10'] = evaluation_df.apply(
    lambda row: recall_at_k(
        row['top10_predicted'],
        row['actual_apps'],
        k=10
    ),
    axis=1
)

evaluation_df['ndcg@10'] = evaluation_df.apply(
    lambda row: ndcg_at_k(
        row['top10_predicted'],
        row['actual_apps'],
        k=10
    ),
    axis=1
)


# Extra diagnostics
evaluation_df['actual_app_count'] = (
    evaluation_df['actual_apps'].apply(len)
)

evaluation_df['hits@10'] = evaluation_df.apply(
    lambda row: len(
        set(row['top10_predicted'][:10])
        & set(row['actual_apps'])
    ),
    axis=1
)


# ------------------------------------------------------------
# 7. POPULARITY BASELINE
#
# IMPORTANT:
# Popularity is calculated from TRAIN ONLY.
# ------------------------------------------------------------

popular_top10 = (
    train_interactions['app_id']
    .value_counts()
    .head(10)
    .index
    .tolist()
)

print("Most popular train apps:")
print(popular_top10)


evaluation_df['popularity_top10'] = [
    popular_top10.copy()
    for _ in range(len(evaluation_df))
]


evaluation_df['pop_recall@10'] = evaluation_df.apply(
    lambda row: recall_at_k(
        row['popularity_top10'],
        row['actual_apps'],
        k=10
    ),
    axis=1
)


evaluation_df['pop_ndcg@10'] = evaluation_df.apply(
    lambda row: ndcg_at_k(
        row['popularity_top10'],
        row['actual_apps'],
        k=10
    ),
    axis=1
)


# ------------------------------------------------------------
# 8. FINAL COMPARISON
# ------------------------------------------------------------

comparison = pd.DataFrame({

    'model': [
        'LambdaMART',
        'Popularity'
    ],

    'Recall@10': [
        evaluation_df['recall@10'].mean(),
        evaluation_df['pop_recall@10'].mean()
    ],

    'NDCG@10': [
        evaluation_df['ndcg@10'].mean(),
        evaluation_df['pop_ndcg@10'].mean()
    ]
})


print("\nOFFICIAL-STYLE EVALUATION")
print("=" * 50)

print(
    "Users evaluated:",
    len(evaluation_df)
)

print(
    "Average actual future apps:",
    evaluation_df['actual_app_count'].mean()
)

print(
    "Average LambdaMART hits@10:",
    evaluation_df['hits@10'].mean()
)

print(
    "LambdaMART zero-recall users:",
    (evaluation_df['recall@10'] == 0).mean()
)

print()

print(comparison)


# ------------------------------------------------------------
# 9. OPTIONAL: relative uplift vs popularity
# ------------------------------------------------------------

lambda_recall = evaluation_df['recall@10'].mean()
pop_recall = evaluation_df['pop_recall@10'].mean()

lambda_ndcg = evaluation_df['ndcg@10'].mean()
pop_ndcg = evaluation_df['pop_ndcg@10'].mean()


if pop_recall > 0:
    recall_uplift = (
        lambda_recall / pop_recall - 1
    ) * 100

    print(
        f"\nRecall@10 uplift vs popularity: "
        f"{recall_uplift:.2f}%"
    )


if pop_ndcg > 0:
    ndcg_uplift = (
        lambda_ndcg / pop_ndcg - 1
    ) * 100

    print(
        f"NDCG@10 uplift vs popularity: "
        f"{ndcg_uplift:.2f}%"
    )


# ------------------------------------------------------------
# 10. Inspect example users
# ------------------------------------------------------------

evaluation_df[
    [
        'user_id',
        'actual_apps',
        'top10_predicted',
        'actual_app_count',
        'hits@10',
        'recall@10',
        'ndcg@10'
    ]
].head(20)

Most popular train apps:
['a_0273', 'a_0327', 'a_0065', 'a_0377', 'a_0241', 'a_0245', 'a_0498', 'a_0477', 'a_0323', 'a_0275']

OFFICIAL-STYLE EVALUATION
Users evaluated: 9489
Average actual future apps: 13.01918010327748
Average LambdaMART hits@10: 0.8856570766150279
LambdaMART zero-recall users: 0.5544314469385604

        model  Recall@10   NDCG@10
0  LambdaMART   0.058411  0.102540
1  Popularity   0.077869  0.108065

Recall@10 uplift vs popularity: -24.99%
NDCG@10 uplift vs popularity: -5.11%


,user_id,actual_apps,top10_predicted,actual_app_count,hits@10,recall@10,ndcg@10
0,u_000000,"[a_0061, a_0092, a_0069]","[a_0127, a_0225, a_0126, a_0337, a_0211, a_039...",3,0,0.000000,0.000000
1,u_000001,"[a_0186, a_0387, a_0316, a_0278, a_0168, a_0484]","[a_0493, a_0241, a_0215, a_0245, a_0065, a_032...",6,0,0.000000,0.000000
2,u_000002,"[a_0095, a_0180, a_0204, a_0172, a_0326, a_042...","[a_0065, a_0327, a_0241, a_0000, a_0039, a_003...",8,0,0.000000,0.000000
3,u_000003,"[a_0277, a_0104, a_0355, a_0191, a_0322, a_038...","[a_0241, a_0327, a_0273, a_0245, a_0065, a_037...",51,4,0.078431,0.457675
4,u_000004,"[a_0266, a_0037, a_0085, a_0065, a_0165, a_012...","[a_0275, a_0273, a_0437, a_0037, a_0241, a_004...",15,2,0.133333,0.158409
5,u_000005,"[a_0377, a_0241, a_0162, a_0181, a_0254, a_036...","[a_0241, a_0215, a_0046, a_0323, a_0150, a_027...",11,1,0.090909,0.220092
6,u_000006,"[a_0244, a_0324, a_0487, a_0254]","[a_0037, a_0048, a_0477, a_0228, a_0054, a_040...",4,0,0.000000,0.000000
7,u_000009,"[a_0201, a_0277, a_0234, a_0355, a_0340, a_019...","[a_0327, a_0241, a_0273, a_0245, a_0065, a_037...",28,1,0.035714,0.066254
8,u_000010,"[a_0240, a_0037, a_0372, a_0084, a_0370, a_004...","[a_0493, a_0037, a_0065, a_0273, a_0327, a_024...",13,1,0.076923,0.138862
9,u_000011,"[a_0277, a_0104, a_0191, a_0016, a_0476, a_009...","[a_0327, a_0241, a_0273, a_0377, a_0245, a_006...",92,7,0.076087,0.742125
